In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l5.2 Optimizers

> 🎯 **Goal:** Understand how an optimizer converts gradients into actual weight changes, and why AdamW is the reliable default for training language models.

**Why this matters.** Backpropagation gives you a gradient: for every weight, a number saying which direction and how steeply the loss would change if you nudged that weight. But the gradient alone does not update anything. The optimizer is the part that decides how big a step to take in that direction, for each weight, on each step. Choose the optimizer poorly and training crawls or diverges. Choose AdamW and, for a from-scratch GPT, it mostly just works.

**The intuition.** Picture our hiker from the last lesson descending the loss mountain. A naive optimizer (plain SGD, stochastic gradient descent) is a hiker who looks at the slope under their feet and takes one fixed-size step downhill, over and over. AdamW is a smarter hiker with two upgrades. First, momentum: like a heavy ball rolling, it keeps some speed from previous steps, so it powers through small bumps and noisy patches instead of jittering. Second, per-direction step sizes: on a path that has been consistently steep it strides confidently, and on a path that keeps changing direction it shortens its stride and treads carefully.

**The mechanics.** AdamW maintains two running averages for every parameter. The first moment is a smoothed average of recent gradients (the momentum, the usual direction of travel). The second moment is a smoothed average of recent squared gradients (a measure of how noisy or large that parameter's gradient has been). The update divides the first moment by the square root of the second: parameters with small, steady gradients get a relatively larger step, and parameters with big, erratic gradients get damped. The "W" adds decoupled weight decay: on every step it also pulls each weight slightly toward zero. This is regularization, a gentle pressure that discourages the model from leaning too hard on any single weight, which helps it generalize. We pass `lr=3e-4` (the base learning rate, the overall step-size dial) and `weight_decay=0.1`.

In [ ]:
from pocketlm import CharTokenizer
corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
data = torch.tensor(tok.encode(corpus), dtype=torch.long)

from pocketlm import PocketLM, PocketLMConfig, init_weights, make_batch
torch.manual_seed(0)
cfg = PocketLMConfig(vocab_size=tok.vocab_size, block_size=32, n_layer=2,
                     n_head=2, n_embd=64)
model = init_weights(PocketLM(cfg))
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
g = torch.Generator().manual_seed(0)
first = None
steps = 60 if os.environ.get("POCKETLM_CI") else 300
for _ in range(steps):
    x, y = make_batch(data, 32, 16, generator=g)
    _, loss = model(x, y)
    if first is None:
        first = loss.item()
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
print("loss:", round(first, 3), "->", round(loss.item(), 3))

**What just happened.** The printout shows the loss at the first step and after the training loop: the second number should be clearly smaller. Each pass through the loop is one step: grab a fresh batch (a small random chunk of the data), compute the loss, call `loss.backward()` to fill in the gradients, then `opt.step()` to apply AdamW's update. The steady drop is proof the optimizer is steering the weights downhill. (Note the `steps` variable shrinks under CI so the lesson runs fast in automated tests; locally it runs longer for a clearer drop.)

⚠️ **Common pitfalls**
- Forgetting `opt.zero_grad()` before `backward()`: PyTorch accumulates gradients by default, so leftover gradients from the previous step pile on and corrupt the update.
- Setting the learning rate far too high (say `lr=1.0`): steps overshoot the valley, the loss bounces or diverges to NaN. Too low (say `lr=1e-8`) and it barely moves.
- Treating AdamW as magic. It is robust, but it cannot rescue a broken init, a bad data pipeline, or a learning rate off by orders of magnitude.

🏋️ **Try it yourself.** Swap `torch.optim.AdamW` for `torch.optim.SGD(model.parameters(), lr=3e-4)` and rerun. With the same learning rate, plain SGD usually drops the loss far less over the same number of steps, a concrete feel for what Adam's adaptive moments buy you. Then push the AdamW `lr` to `3e-2` and watch it become unstable.

In [ ]:
assert loss.item() < first